# LeetCode 1754 — Largest Merge Of Two Strings bằng Python

Link đề: https://leetcode.com/problems/largest-merge-of-two-strings/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def largestMerge(self, word1: str, word2: str) -> str:
        ...
```

## 1. Mục tiêu học

1. Hiểu **thứ tự từ điển** (lexicographic) của chuỗi trong Python, kể cả trường hợp một chuỗi là **tiền tố** của chuỗi kia.
2. Viết vét cạn mọi cách **trộn** (interleave) hai chuỗi, có ghi nhớ theo `(i, j)`, làm đáp án chuẩn.
3. Nhận ra bẫy của tham lam "so ký tự đầu": khi hai ký tự đầu **bằng nhau** phải nhìn xa hơn — so sánh **cả hậu tố** `word1[i:]` với `word2[j:]`.
4. Cài so sánh hậu tố không cắt chuỗi (hai con trỏ) và bản nâng cao dùng **mảng hậu tố** (suffix array) để so sánh `O(1)`.
5. Chứng minh tham lam bằng lập luận **hoán đổi** và kiểm thử đối chiếu ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho hai chuỗi `word1`, `word2`. Xây chuỗi `merge` như sau: chừng nào còn một trong hai chuỗi khác rỗng, chọn **một** trong hai thao tác:
- nếu `word1` khác rỗng: nối ký tự **đầu** của `word1` vào cuối `merge` và xóa nó khỏi `word1`;
- nếu `word2` khác rỗng: làm tương tự với `word2`.

Trả về `merge` **lớn nhất theo thứ tự từ điển**. (Chuỗi `a` lớn hơn `b` cùng độ dài nếu ở vị trí khác nhau đầu tiên, ký tự của `a` lớn hơn.)

**Ví dụ 1:** `word1 = "cabaa"`, `word2 = "bcaaa"` → `"cbcabaaaaa"`.
Lấy `c` (word1) → `b` (word2) → `c` (word2) → `a` (word1) → `b` (word1) → nối 5 chữ `a` còn lại.

**Ví dụ 2:** `word1 = "abcabc"`, `word2 = "abdcaba"` → `"abdcabcabcaba"`.

**Ràng buộc:** `1 ≤ word1.length, word2.length ≤ 3000`; chỉ gồm chữ thường.

In [1]:
import random
import time
from functools import lru_cache

examples = [("cabaa", "bcaaa", "cbcabaaaaa"), ("abcabc", "abdcaba", "abdcabcabcaba")]
for w1, w2, exp in examples:
    print(f"word1={w1!r}, word2={w2!r} -> kỳ vọng {exp!r}")

word1='cabaa', word2='bcaaa' -> kỳ vọng 'cbcabaaaaa'
word1='abcabc', word2='abdcaba' -> kỳ vọng 'abdcabcabcaba'


## 3. Kiến thức nền

### 3.1 So sánh chuỗi trong Python

`s < t` so từng ký tự từ trái sang; ở vị trí khác nhau đầu tiên, ký tự nhỏ hơn thì chuỗi nhỏ hơn. Nếu một chuỗi là **tiền tố** của chuỗi kia thì chuỗi **ngắn hơn nhỏ hơn**: `"ab" < "abc"`.

In [2]:
print("abcd" > "abcc", "ab" < "abc", "b" > "abzzz", "" < "a", max(["ca", "cb", "c"]))

True True True True cb


### 3.2 Hậu tố (suffix) và slicing

`s[i:]` là hậu tố bắt đầu tại `i`. Tạo hậu tố tốn `O(len)` thời gian và bộ nhớ, nên so sánh `s[i:] > t[j:]` tốn `O(n + m)` trong trường hợp tệ.

In [3]:
s = "cabaa"
print([s[i:] for i in range(len(s) + 1)])

['cabaa', 'abaa', 'baa', 'aa', 'a', '']


### 3.3 Ghép chuỗi hiệu quả bằng list + `"".join`

Nối `ans += ch` lặp nhiều lần có thể chậm; cách chuẩn: gom ký tự vào list rồi `"".join(list)` một lần ở cuối.

In [4]:
parts = []
for ch in "hello":
    parts.append(ch.upper())
print("".join(parts))

HELLO


### 3.4 So sánh hai hậu tố bằng hai con trỏ (không cắt chuỗi)

Đi song song `p = i`, `q = j` khi `s[p] == t[q]`; dừng ở vị trí khác nhau hoặc khi một chuỗi hết.

In [5]:
def suffix_greater(s, i, t, j):
    """True nếu s[i:] > t[j:] (theo thứ tự từ điển của Python)."""
    while i < len(s) and j < len(t):
        if s[i] != t[j]:
            return s[i] > t[j]
        i += 1; j += 1
    return i < len(s)        # t hết trước (t[j:] là tiền tố của s[i:]) => s lớn hơn

for (s, i, t, j) in [("cabaa", 0, "bcaaa", 0), ("abaa", 1, "caaa", 1), ("ab", 0, "abc", 0), ("abc", 0, "ab", 0)]:
    assert suffix_greater(s, i, t, j) == (s[i:] > t[j:])
    print(repr(s[i:]), ">", repr(t[j:]), "?", suffix_greater(s, i, t, j))

'cabaa' > 'bcaaa' ? True
'baa' > 'aaa' ? True
'ab' > 'abc' ? False
'abc' > 'ab' ? True


## 4. Ý tưởng nền tảng / trực giác

Muốn chuỗi lớn nhất theo từ điển ⇒ ký tự **đầu tiên** phải lớn nhất có thể, rồi ký tự thứ hai, … Ở mỗi bước, ta chỉ có hai lựa chọn: ký tự đầu của `word1` hoặc của `word2`.

- Nếu hai ký tự đầu **khác nhau** ⇒ hiển nhiên lấy ký tự lớn hơn.
- Nếu **bằng nhau**? Ví dụ `word1 = "ab"`, `word2 = "ac"`: lấy `a` từ đâu cũng ra `a` ngay, nhưng **tương lai** khác nhau:
  - lấy từ `word1` ⇒ còn `"b"` và `"ac"` ⇒ tốt nhất `"acb"`… ⇒ tổng `"aacb"`;
  - lấy từ `word2` ⇒ còn `"ab"` và `"c"` ⇒ tốt nhất `"cab"` ⇒ tổng `"acab"`.

`"acab" > "aacb"` ⇒ nên lấy từ `word2` — chuỗi có **phần sau lớn hơn** (`"ac" > "ab"`). Quy tắc: lấy từ chuỗi có **hậu tố lớn hơn**.

In [6]:
def all_merges(a, b):
    if not a: return [b]
    if not b: return [a]
    return [a[0] + m for m in all_merges(a[1:], b)] + [b[0] + m for m in all_merges(a, b[1:])]

ms = all_merges("ab", "ac")
print(sorted(set(ms), reverse=True))

['acab', 'abac', 'aacb', 'aabc']


## 5. Cách 1 — Brute Force / baseline: thử mọi cách trộn (có ghi nhớ)

**Ý tưởng:** `best(i, j)` = merge lớn nhất tạo được từ `word1[i:]` và `word2[j:]`:
- nếu một chuỗi hết: trả phần còn lại của chuỗi kia;
- ngược lại: `max(word1[i] + best(i+1, j), word2[j] + best(i, j+1))`.

Không ghi nhớ thì có `C(n+m, n)` cách trộn (hàm mũ). Ghi nhớ theo `(i, j)` còn `O(n·m)` trạng thái, mỗi trạng thái ghép chuỗi `O(n+m)`.

In [7]:
def merge_bruteforce(word1, word2):
    @lru_cache(maxsize=None)
    def best(i, j):
        if i == len(word1): return word2[j:]
        if j == len(word2): return word1[i:]
        return max(word1[i] + best(i + 1, j), word2[j] + best(i, j + 1))
    return best(0, 0)

for w1, w2, exp in examples:
    got = merge_bruteforce(w1, w2)
    print(got, got == exp); assert got == exp

cbcabaaaaa True
abdcabcabcaba True


**Độ phức tạp:** Time `O(n·m·(n+m))`, Space `O(n·m·(n+m))` cho các chuỗi được ghi nhớ — với `n = m = 3000` là ~`5·10¹⁰`: **không thể**. Còn bị giới hạn độ sâu đệ quy.

**Nút thắt:** ta so sánh **toàn bộ** hai kết quả tương lai, trong khi chỉ cần biết **hậu tố nào lớn hơn** để quyết định.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** tại trạng thái `(i, j)`:
- **Nếu** `word1[i:] > word2[j:]` **thì** lấy `word1[i]`.
- **Ngược lại** lấy `word2[j]` (khi hai hậu tố bằng nhau thì lấy bên nào cũng như nhau).

Vì sao so **cả hậu tố** mà không chỉ ký tự đầu? Khi ký tự đầu bằng nhau, lấy ký tự từ chuỗi có hậu tố lớn hơn sẽ "mở khóa" ký tự lớn ở phía sau sớm hơn, trong khi chuỗi kia vẫn giữ nguyên để dùng sau.

**Invariant:** sau mỗi bước, phần `merge` đã xây là **tiền tố** của merge tối ưu.

Chi phí: mỗi bước so sánh hậu tố `O(n+m)`, tổng `O((n+m)²) ≈ 3,6·10⁷` phép so ký tự — so sánh chuỗi trong Python được cài bằng C nên vẫn nhanh.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam so sánh hậu tố bằng slicing

**Giải thích từng đoạn:**
1. `i = j = 0`, `ans = []`.
2. Khi cả hai còn ký tự: nếu `word1[i:] > word2[j:]` thì lấy `word1[i]`, ngược lại lấy `word2[j]`.
3. Nối phần còn lại của hai chuỗi (một trong hai đã rỗng).

In [8]:
def merge_greedy_slice(word1, word2):
    i = j = 0
    ans = []
    while i < len(word1) and j < len(word2):
        if word1[i:] > word2[j:]:
            ans.append(word1[i]); i += 1
        else:
            ans.append(word2[j]); j += 1
    ans.append(word1[i:]); ans.append(word2[j:])
    return "".join(ans)

for w1, w2, exp in examples:
    assert merge_greedy_slice(w1, w2) == exp
print([merge_greedy_slice(a, b) for a, b, _ in examples])

['cbcabaaaaa', 'abdcabcabcaba']


**Code nộp LeetCode (Cách 2):** xem mục 9 (bản được chọn để nộp).

**Độ phức tạp Cách 2:** Time `O((n+m)²)` tệ nhất (mỗi bước cắt và so hậu tố), Space `O(n+m)`.

### 7.2 Cách 3 — Tham lam, so hậu tố bằng hai con trỏ (không tạo chuỗi con)

Cùng ý tưởng, nhưng dùng `suffix_greater` (mục 3.4) để không cấp phát hậu tố mới — hữu ích khi viết ở ngôn ngữ khác hoặc khi bộ nhớ hạn chế.

In [9]:
class SolutionPointers:
    def largestMerge(self, word1: str, word2: str) -> str:
        n, m = len(word1), len(word2)

        def greater(i: int, j: int) -> bool:
            while i < n and j < m:
                if word1[i] != word2[j]:
                    return word1[i] > word2[j]
                i += 1
                j += 1
            return i < n

        i = j = 0
        ans = []
        while i < n and j < m:
            if greater(i, j):
                ans.append(word1[i]); i += 1
            else:
                ans.append(word2[j]); j += 1
        return "".join(ans) + word1[i:] + word2[j:]

print([SolutionPointers().largestMerge(a, b) for a, b, _ in examples])

['cbcabaaaaa', 'abdcabcabcaba']


**Độ phức tạp Cách 3:** Time `O((n+m)²)` tệ nhất (vòng `greater` viết bằng Python nên chậm hơn Cách 2 về hằng số), Space `O(1)` ngoài kết quả.

### 7.3 Cách 4 — Mảng hậu tố (suffix array): so sánh hậu tố trong `O(1)`

Ghép `T = word1 + "0" + word2 + "1"` (`'0' < '1' < 'a'`). So sánh hậu tố của `T` tại `i` (trong `word1`) và tại `n + 1 + j` (trong `word2`) cho **cùng kết quả** với `word1[i:]` vs `word2[j:]`:
- khác nhau ở một vị trí nằm trong cả hai ⇒ như nhau;
- `word1[i:]` là tiền tố thật của `word2[j:]` ⇒ bên `T` gặp `'0'` so với một chữ cái ⇒ bé hơn ✓;
- `word2[j:]` là tiền tố thật của `word1[i:]` ⇒ gặp `'1'` so với chữ cái ⇒ bé hơn ✓;
- bằng nhau ⇒ `'0' < '1'` ⇒ chọn `word2` — vô hại vì hai lựa chọn như nhau.

Tính `rank` của mọi hậu tố bằng **nhân đôi** (prefix doubling): sắp theo cặp `(rank[k], rank[k + len])`, nhân đôi `len` tới khi mọi rank khác nhau. Sau đó mỗi bước tham lam chỉ so hai số nguyên.

In [10]:
def suffix_ranks(T):
    n = len(T)
    rank = [ord(c) for c in T]
    k = 1
    while True:
        key = [(rank[i], rank[i + k] if i + k < n else -1) for i in range(n)]
        order = sorted(range(n), key=lambda i: key[i])
        new = [0] * n
        for t in range(1, n):
            new[order[t]] = new[order[t - 1]] + (key[order[t]] != key[order[t - 1]])
        rank = new
        if rank[order[-1]] == n - 1:
            return rank
        k *= 2

def merge_suffix_array(word1, word2):
    n, m = len(word1), len(word2)
    rk = suffix_ranks(word1 + "0" + word2 + "1")
    i = j = 0; ans = []
    while i < n and j < m:
        if rk[i] > rk[n + 1 + j]:
            ans.append(word1[i]); i += 1
        else:
            ans.append(word2[j]); j += 1
    return "".join(ans) + word1[i:] + word2[j:]

T = "ab0ac1"
rk = suffix_ranks(T)
print("rank các hậu tố của", T, ":", rk, "| thứ tự:", [T[i:] for i in sorted(range(len(T)), key=lambda i: rk[i])])
print([merge_suffix_array(a, b) for a, b, _ in examples])

rank các hậu tố của ab0ac1 : [2, 4, 0, 3, 5, 1] | thứ tự: ['0ac1', '1', 'ab0ac1', 'ac1', 'b0ac1', 'c1']
['cbcabaaaaa', 'abdcabcabcaba']


**Code nộp LeetCode (Cách 4):**

In [11]:
class SolutionSA:
    def largestMerge(self, word1: str, word2: str) -> str:
        n, m = len(word1), len(word2)
        T = word1 + "0" + word2 + "1"
        L = len(T)
        rank = [ord(c) for c in T]
        k = 1
        while True:
            key = [(rank[i], rank[i + k] if i + k < L else -1) for i in range(L)]
            order = sorted(range(L), key=key.__getitem__)
            new = [0] * L
            for t in range(1, L):
                new[order[t]] = new[order[t - 1]] + (key[order[t]] != key[order[t - 1]])
            rank = new
            if rank[order[-1]] == L - 1:
                break
            k *= 2
        i = j = 0
        ans = []
        while i < n and j < m:
            if rank[i] > rank[n + 1 + j]:
                ans.append(word1[i]); i += 1
            else:
                ans.append(word2[j]); j += 1
        return "".join(ans) + word1[i:] + word2[j:]

print([SolutionSA().largestMerge(a, b) for a, b, _ in examples])

['cbcabaaaaa', 'abdcabcabcaba']


**Độ phức tạp Cách 4:** dựng rank `O(L log² L)` (mỗi vòng sắp `O(L log L)`, tối đa `log L` vòng), tham lam `O(n + m)` ⇒ tổng `O(L log² L)`, Space `O(L)`, với `L = n + m + 2`.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1** (`"cabaa"`, `"bcaaa"`):

| bước | word1[i:] | word2[j:] | so sánh | lấy | merge |
|---|---|---|---|---|---|
| 1 | cabaa | bcaaa | `c > b` | w1 `c` | c |
| 2 | abaa | bcaaa | `a < b` | w2 `b` | cb |
| 3 | abaa | caaa | `a < c` | w2 `c` | cbc |
| 4 | abaa | aaa | `ab… > aa…` | w1 `a` | cbca |
| 5 | baa | aaa | `b > a` | w1 `b` | cbcab |
| … | aa | aaa | `aa < aaa` | w2 `a` | … |

In [12]:
def trace_merge(word1, word2):
    i = j = 0; ans = []; step = 0
    while i < len(word1) and j < len(word2):
        step += 1
        s1, s2 = word1[i:], word2[j:]
        if s1 > s2:
            ans.append(word1[i]); i += 1; who = "word1"
        else:
            ans.append(word2[j]); j += 1; who = "word2"
        print(f"bước {step}: {s1!r:>9} vs {s2!r:<9} -> lấy {ans[-1]!r} từ {who} | merge={''.join(ans)!r}")
    rest = word1[i:] + word2[j:]
    print("nối phần còn lại:", repr(rest))
    res = "".join(ans) + rest
    print("Kết quả:", res)
    return res

assert trace_merge(*examples[0][:2]) == examples[0][2]

bước 1:   'cabaa' vs 'bcaaa'   -> lấy 'c' từ word1 | merge='c'
bước 2:    'abaa' vs 'bcaaa'   -> lấy 'b' từ word2 | merge='cb'
bước 3:    'abaa' vs 'caaa'    -> lấy 'c' từ word2 | merge='cbc'
bước 4:    'abaa' vs 'aaa'     -> lấy 'a' từ word1 | merge='cbca'
bước 5:     'baa' vs 'aaa'     -> lấy 'b' từ word1 | merge='cbcab'
bước 6:      'aa' vs 'aaa'     -> lấy 'a' từ word2 | merge='cbcaba'
bước 7:      'aa' vs 'aa'      -> lấy 'a' từ word2 | merge='cbcabaa'
bước 8:      'aa' vs 'a'       -> lấy 'a' từ word1 | merge='cbcabaaa'
bước 9:       'a' vs 'a'       -> lấy 'a' từ word2 | merge='cbcabaaaa'
nối phần còn lại: 'a'
Kết quả: cbcabaaaaa


Trace ví dụ tự dựng `("ab", "ac")` — hai ký tự đầu bằng nhau (ví dụ tự dựng, đối chiếu brute force; nên đối chiếu lại đề gốc):

In [13]:
assert trace_merge("ab", "ac") == merge_bruteforce("ab", "ac") == "acab"

bước 1:      'ab' vs 'ac'      -> lấy 'a' từ word2 | merge='a'
bước 2:      'ab' vs 'c'       -> lấy 'c' từ word2 | merge='ac'
nối phần còn lại: 'ab'
Kết quả: acab


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — tham lam so sánh hậu tố bằng slicing (ngắn, nhanh trong Python vì so sánh chuỗi chạy bằng C).

In [14]:
class Solution:
    def largestMerge(self, word1: str, word2: str) -> str:
        i = j = 0
        ans = []
        while i < len(word1) and j < len(word2):
            if word1[i:] > word2[j:]:
                ans.append(word1[i])
                i += 1
            else:
                ans.append(word2[j])
                j += 1
        return "".join(ans) + word1[i:] + word2[j:]

for w1, w2, exp in examples:
    got = Solution().largestMerge(w1, w2)
    print(got, got == exp); assert got == exp

cbcabaaaaa True
abdcabcabcaba True


## 10. Vì sao thuật toán đúng

Ký hiệu `A = word1[i:]`, `B = word2[j:]`, `F(A, B)` = merge lớn nhất. Ta chứng minh: nếu `A > B` thì `F(A, B) = A[0] + F(A[1:], B)`.

**Trường hợp 1: `A[0] ≠ B[0]`.** Vì `A > B` nên `A[0] > B[0]`. Mọi merge bắt đầu bằng `B[0]` đều thua mọi merge bắt đầu bằng `A[0]` ngay ở ký tự đầu ⇒ phải lấy `A[0]`.

**Trường hợp 2: `A[0] = B[0]`.** Gọi `k` là vị trí khác nhau đầu tiên (nếu `B` là tiền tố thật của `A` thì `k = len(B)`); `A[:k] = B[:k] = P` và `A[k] > B[k]` (hoặc `B` đã hết). *Phác thảo lập luận hoán đổi:* lấy một merge tối ưu `M` bắt đầu bằng ký tự của `B`. Trong `M`, chừng nào cả hai chuỗi mới tiêu thụ ít hơn `k` ký tự, ta có thể **đổi nhãn** "lấy từ A" ↔ "lấy từ B" cho các ký tự trong phần chung `P` mà chuỗi ghi ra không đổi (vì `A[t] = B[t]` với `t < k`). Sau khi đổi nhãn, merge mới bắt đầu bằng ký tự của `A`, và tại thời điểm một bên chạm vị trí `k`, bên chạm là `A` (đã có `A[k]` lớn hơn sẵn sàng) thay vì `B` (với `B[k]` nhỏ hơn hoặc đã hết). Có ký tự lớn hơn sẵn sàng không bao giờ làm merge tối ưu phía sau nhỏ đi. Vì vậy luôn tồn tại một merge tối ưu bắt đầu bằng `A[0]`, đúng như tham lam chọn.

(Chứng minh chặt chẽ đầy đủ khá dài; ở đây ta bổ sung **kiểm chứng thực nghiệm**: mục 12 đối chiếu tham lam với vét cạn trên hàng nghìn cặp chuỗi ngẫu nhiên với bảng chữ nhỏ — nơi các trường hợp hòa xuất hiện dày đặc.)

**Khi `A = B`:** hai lựa chọn đối xứng hoàn toàn ⇒ kết quả như nhau.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(word1)`, `m = len(word2)`, `N = n + m ≤ 6000`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn ghi nhớ | `O(n·m·N)` | `O(n·m·N)` |
| 2. Tham lam + slicing | `O(N²)` tệ nhất (so sánh bằng C) | `O(N)` |
| 3. Tham lam + hai con trỏ | `O(N²)` tệ nhất (vòng Python) | `O(1)` phụ |
| 4. Suffix array | `O(N log² N)` | `O(N)` |

Trường hợp tệ cho Cách 2/3: hai chuỗi gần như toàn chữ `a` (so sánh phải đi rất xa). Đo:

In [15]:
w1 = "a" * 2999 + "b"; w2 = "a" * 2999 + "c"
for name, f in [("Cách 2", Solution().largestMerge), ("Cách 3", SolutionPointers().largestMerge), ("Cách 4", SolutionSA().largestMerge)]:
    t = time.perf_counter(); r = f(w1, w2); dt = time.perf_counter() - t
    print(f"{name}: len={len(r)}, đuôi={r[-3:]!r} ({dt*1000:.0f} ms)")

Cách 2: len=6000, đuôi='aab' (0 ms)


Cách 3: len=6000, đuôi='aab' (230 ms)
Cách 4: len=6000, đuôi='aab' (15 ms)


## 12. Bộ test

Đối chiếu tham lam (3 cách) với vét cạn trên nhiều cặp chuỗi ngẫu nhiên ngắn (seed cố định, bảng chữ `{a,b}` / `{a,b,c}`), thêm edge case và một test lớn đúng giới hạn.

In [16]:
ALL = {"slice": merge_greedy_slice, "SA": merge_suffix_array,
       "SolPointers": lambda a, b: SolutionPointers().largestMerge(a, b),
       "SolSA": lambda a, b: SolutionSA().largestMerge(a, b),
       "submit": lambda a, b: Solution().largestMerge(a, b)}
def run_all(a, b):
    res = {k: f(a, b) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (a, b, res)
    return next(iter(res.values()))

for w1, w2, exp in examples:
    assert run_all(w1, w2) == exp
edge = [("a", "a", "aa"), ("a", "b", "ba"), ("ab", "ab", "abab"), ("ab", "ac", "acab"),
        ("aa", "aab", "aabaa"), ("z", "zzzz", "zzzzz"), ("ba", "b", "bba")]
for a, b, exp in edge:
    assert merge_bruteforce(a, b) == exp, (a, b)
    assert run_all(a, b) == exp
rng = random.Random(1754)
for _ in range(3000):
    alpha = rng.choice(["ab", "abc"])
    a = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 7)))
    b = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 7)))
    assert run_all(a, b) == merge_bruteforce(a, b), (a, b)
big1 = "".join(rng.choice("ab") for _ in range(3000)); big2 = "".join(rng.choice("ab") for _ in range(3000))
r = run_all(big1, big2)
assert sorted(r) == sorted(big1 + big2)
print("Ví dụ, edge case, 3000 test ngẫu nhiên và test lớn: OK")

Ví dụ, edge case, 3000 test ngẫu nhiên và test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ so sánh ký tự đầu, hòa thì lấy `word1`

In [17]:
def wrong_first_char(word1, word2):
    i = j = 0; ans = []
    while i < len(word1) and j < len(word2):
        if word1[i] >= word2[j]:                  # SAI khi word1[i] == word2[j]
            ans.append(word1[i]); i += 1
        else:
            ans.append(word2[j]); j += 1
    return "".join(ans) + word1[i:] + word2[j:]

w1, w2, exp = examples[1]
print("Ví dụ 2 — sai :", wrong_first_char(w1, w2))
print("Ví dụ 2 — đúng:", exp)

Ví dụ 2 — sai : abcabcabdcaba
Ví dụ 2 — đúng: abdcabcabcaba


**Vì sao sai:** ở Ví dụ 2, `"abcabc"` và `"abdcaba"` có chung tiền tố `"ab"`; khi hòa, code sai luôn lấy từ `word1`, nên chữ `d` của `word2` bị đẩy ra sau. Ví dụ ngắn gọn hơn là `("ab", "ac")`: code sai cho `"abac"` thay vì `"acab"`. (Ở Ví dụ 1 code sai lại tình cờ ra đúng — một lý do khiến lỗi này khó phát hiện nếu chỉ test ví dụ đầu.)

In [18]:
print("sai :", wrong_first_char("ab", "ac"), "| đúng:", Solution().largestMerge("ab", "ac"))
w1, w2, exp = examples[0]
print("Ví dụ 1 — code sai vẫn ra:", wrong_first_char(w1, w2), "== đúng?", wrong_first_char(w1, w2) == exp)

sai : abac | đúng: acab
Ví dụ 1 — code sai vẫn ra: cbcabaaaaa == đúng? True


**Sửa:** so sánh **cả hậu tố** `word1[i:] > word2[j:]`.

### Lỗi 2 — Hòa ký tự đầu thì so ký tự **tiếp theo** (chỉ nhìn thêm 1 bước)

In [19]:
def wrong_peek_one(word1, word2):
    i = j = 0; ans = []
    while i < len(word1) and j < len(word2):
        if word1[i] != word2[j]:
            take1 = word1[i] > word2[j]
        else:
            take1 = word1[i + 1:i + 2] >= word2[j + 1:j + 2]      # SAI: chỉ nhìn 1 ký tự
        if take1:
            ans.append(word1[i]); i += 1
        else:
            ans.append(word2[j]); j += 1
    return "".join(ans) + word1[i:] + word2[j:]

a, b = "aab", "aac"            # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_peek_one(a, b), "| đúng:", Solution().largestMerge(a, b), "| brute:", merge_bruteforce(a, b))

sai : aabaac | đúng: aacaab | brute: aacaab


**Vì sao sai:** phần bằng nhau có thể dài tùy ý; phải đi tới vị trí khác nhau **đầu tiên**. **Sửa:** so cả hậu tố.

### Lỗi 3 — Quên trường hợp một hậu tố là **tiền tố** của hậu tố kia (khi tự viết hàm so sánh)

In [20]:
def wrong_prefix_case(word1, word2):
    def greater(i, j):
        while i < len(word1) and j < len(word2):
            if word1[i] != word2[j]:
                return word1[i] > word2[j]
            i += 1; j += 1
        return False                               # SAI: nếu word2 hết trước thì word1 lớn hơn
    i = j = 0; ans = []
    while i < len(word1) and j < len(word2):
        if greater(i, j):
            ans.append(word1[i]); i += 1
        else:
            ans.append(word2[j]); j += 1
    return "".join(ans) + word1[i:] + word2[j:]

a, b = "ab", "a"               # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_prefix_case(a, b), "| đúng:", Solution().largestMerge(a, b), "| brute:", merge_bruteforce(a, b))

sai : aab | đúng: aba | brute: aba


**Vì sao sai:** `"ab" > "a"` (cùng tiền tố thì chuỗi dài hơn lớn hơn), nên phải lấy `a` từ `word1` để chữ `b` lộ ra sớm: `"a" + "b" + "a" = "aba"`. Hàm sai trả `False` khi `word2` hết trước, nên lấy `a` từ `word2`, chữ `b` bị đẩy xuống cuối: `"aab"`. **Sửa:** `return i < len(word1)`.

### Lỗi 4 — Nối chuỗi trong vòng lặp bằng `ans = ans + ch` và tạo hậu tố không cần thiết

Không sai kết quả nhưng chậm hơn: tạo chuỗi mới mỗi bước. Với `N = 6000` vẫn qua, nhưng thói quen tốt là dùng list + `join`.

In [21]:
def slow_concat(word1, word2):
    i = j = 0; ans = ""
    while i < len(word1) and j < len(word2):
        if word1[i:] > word2[j:]:
            ans = ans + word1[i]; i += 1
        else:
            ans = ans + word2[j]; j += 1
    return ans + word1[i:] + word2[j:]

for f in (slow_concat, Solution().largestMerge):
    t = time.perf_counter(); f(big1, big2); print(getattr(f, "__name__", "?"), f"{(time.perf_counter()-t)*1000:.0f} ms")

slow_concat 3 ms
largestMerge 2 ms


Ở đây chênh lệch nhỏ vì CPython đôi khi tối ưu được phép nối chuỗi tại chỗ — nhưng đó là chi tiết cài đặt, không được đảm bảo (PyPy hay ngôn ngữ khác sẽ thành `O(N²)` cấp phát). **Sửa:** gom vào list rồi `"".join`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuỗi lớn nhất / nhỏ nhất theo từ điển, mỗi bước chọn một trong hai nguồn" ⇒ tham lam theo **hậu tố**.
- Hòa ở ký tự hiện tại ⇒ nghĩ ngay tới "nhìn xa đến vị trí khác nhau đầu tiên".
- Nhớ quy ước tiền tố: chuỗi ngắn hơn **nhỏ hơn**.
- Cần so sánh hậu tố nhiều lần ⇒ suffix array / hashing + nhị phân để `O(1)` hoặc `O(log)` mỗi lần.
- Luôn kiểm bằng vét cạn trên bảng chữ **nhỏ** (`{a,b}`) — nơi các trường hợp hòa xuất hiện dày đặc.

## 15. Biến thể của bài

1. **Merge nhỏ nhất** theo từ điển: đổi dấu so sánh, nhưng cẩn thận quy ước tiền tố — thêm ký tự "lớn" vào cuối mỗi chuỗi (ví dụ `'{'`) rồi so sánh (Bài tập 5).
2. **321 Create Maximum Number**: chọn `k` chữ số từ hai mảng giữ thứ tự, bước "merge" dùng đúng tham lam hậu tố này.
3. **Nhiều hơn hai chuỗi**: mỗi bước lấy từ chuỗi có hậu tố lớn nhất.
4. **1673 Find the Most Competitive Subsequence**, **402 Remove K Digits**: tham lam từ điển với stack.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn ghi nhớ | `max` của hai lựa chọn | `O(nmN)` | `O(nmN)` | Kiểm chứng, chuỗi ngắn |
| 2. Tham lam + slicing | So `word1[i:] > word2[j:]` | `O(N²)` (nhanh nhờ C) | `O(N)` | **Bản nộp khuyên dùng** |
| 3. Tham lam + hai con trỏ | So hậu tố không cắt chuỗi | `O(N²)` | `O(1)` | Ngôn ngữ khác, hạn chế bộ nhớ |
| 4. Suffix array | Rank hậu tố, so `O(1)` | `O(N log² N)` | `O(N)` | `N` rất lớn (10⁵+) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `("a","b")`; (b) `("ba","ab")`; (c) `("aaa","aa")`; (d) `("abc","abd")`.

**Bài 2 (tính tay):** Trace tham lam cho `("bab", "bb")`: ghi `word1[i:]`, `word2[j:]` mỗi bước và kết quả.

**Bài 3 (điền TODO):** Hoàn thiện `suffix_greater_todo(s, i, t, j)` (so sánh hậu tố bằng hai con trỏ, xử lý đúng trường hợp tiền tố).

**Bài 4 (tự viết lại):** Viết `merge_with_labels(word1, word2)` trả về `(merge, labels)` với `labels` là chuỗi gồm `'1'`/`'2'` cho biết mỗi ký tự lấy từ đâu; kiểm tra tính hợp lệ.

**Bài 5 (mini challenge):** Viết `smallest_merge(word1, word2)` trả merge **nhỏ nhất** theo từ điển, đối chiếu vét cạn.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def suffix_greater_todo(s, i, t, j):
    while i < len(s) and j < len(t):
        # TODO: nếu khác nhau thì trả về s[i] > t[j]; ngược lại tiến cả hai
        pass
    # TODO: một chuỗi đã hết — chuỗi nào còn ký tự thì lớn hơn
    return None

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def merge_with_labels(word1, word2):
    # TODO: như Cách 2, thêm list labels
    pass

def smallest_merge(word1, word2):
    # TODO: so (word1[i:] + '{') < (word2[j:] + '{') rồi lấy bên nhỏ hơn
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: so hậu tố. Mức 2: nhớ quy ước tiền tố. Mức 3: (a) `"ba"`; (b) `"baba"`; (c) `"aaaaa"`; (d) `"abdabc"`.

**Bài 2.** Mức 1: `"bab"` vs `"bb"`: `b = b`, rồi `a < b` ⇒ `"bb"` lớn hơn ⇒ lấy từ word2. Mức 2: `"bab"` vs `"b"`: `"bab" > "b"` ⇒ lấy word1. Mức 3: kết quả `"bbbab"`.

**Bài 3.** Mức 1: `if s[i] != t[j]: return s[i] > t[j]`. Mức 2: `i += 1; j += 1`. Mức 3: `return i < len(s)`.

**Bài 4.** Mức 1: thêm `labels.append('1')` khi lấy từ word1. Mức 2: phần còn lại gán nhãn tương ứng. Mức 3: kiểm: lọc ký tự nhãn `'1'` phải ra đúng `word1`, nhãn `'2'` ra `word2`.

**Bài 5.** Mức 1: đổi `>` thành `<`. Mức 2: quy ước tiền tố gây khó: với `("b", "ba")`, so `"b" < "ba"` sẽ lấy `b` từ `word1` trước ⇒ `"bba"`, nhưng lấy từ `word2` trước cho `"bab"` nhỏ hơn. Dùng vét cạn để thấy quy ước tiền tố phải **đảo lại**. Mức 3: thêm ký tự lớn hơn mọi chữ (`'{'`) vào cuối mỗi hậu tố trước khi so.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
b1 = [Solution().largestMerge(a, b) for a, b in [("a", "b"), ("ba", "ab"), ("aaa", "aa"), ("abc", "abd")]]
print("Bài 1:", b1); assert b1 == ["ba", "baba", "aaaaa", "abdabc"]
assert b1 == [merge_bruteforce(a, b) for a, b in [("a", "b"), ("ba", "ab"), ("aaa", "aa"), ("abc", "abd")]]
print("Bài 2:"); assert trace_merge("bab", "bb") == "bbbab"

Bài 1: ['ba', 'baba', 'aaaaa', 'abdabc']
Bài 2:
bước 1:     'bab' vs 'bb'      -> lấy 'b' từ word2 | merge='b'
bước 2:     'bab' vs 'b'       -> lấy 'b' từ word1 | merge='bb'
bước 3:      'ab' vs 'b'       -> lấy 'b' từ word2 | merge='bbb'
nối phần còn lại: 'ab'
Kết quả: bbbab


In [25]:
# Bài 3, 4
def suffix_greater_todo(s, i, t, j):
    while i < len(s) and j < len(t):
        if s[i] != t[j]:
            return s[i] > t[j]
        i += 1; j += 1
    return i < len(s)

def merge_with_labels(word1, word2):
    i = j = 0; ans = []; lab = []
    while i < len(word1) and j < len(word2):
        if suffix_greater_todo(word1, i, word2, j):
            ans.append(word1[i]); lab.append("1"); i += 1
        else:
            ans.append(word2[j]); lab.append("2"); j += 1
    ans += list(word1[i:]) + list(word2[j:]); lab += ["1"] * (len(word1) - i) + ["2"] * (len(word2) - j)
    return "".join(ans), "".join(lab)

rng = random.Random(9)
for _ in range(1500):
    a = "".join(rng.choice("ab") for _ in range(rng.randint(0, 6)))
    b = "".join(rng.choice("ab") for _ in range(rng.randint(0, 6)))
    for i in range(len(a) + 1):
        for j in range(len(b) + 1):
            assert suffix_greater_todo(a, i, b, j) == (a[i:] > b[j:])
    if a and b:
        mg, lab = merge_with_labels(a, b)
        assert mg == Solution().largestMerge(a, b)
        assert "".join(c for c, l in zip(mg, lab) if l == "1") == a and "".join(c for c, l in zip(mg, lab) if l == "2") == b
print("Bài 3, 4: OK | ví dụ:", merge_with_labels("cabaa", "bcaaa"))

Bài 3, 4: OK | ví dụ: ('cbcabaaaaa', '1221122121')


In [26]:
# Bài 5
def smallest_merge(word1, word2):
    i = j = 0; ans = []
    while i < len(word1) and j < len(word2):
        if word1[i:] + "{" < word2[j:] + "{":
            ans.append(word1[i]); i += 1
        else:
            ans.append(word2[j]); j += 1
    return "".join(ans) + word1[i:] + word2[j:]

def smallest_brute(a, b):
    @lru_cache(maxsize=None)
    def best(i, j):
        if i == len(a): return b[j:]
        if j == len(b): return a[i:]
        return min(a[i] + best(i + 1, j), b[j] + best(i, j + 1))
    return best(0, 0)

rng = random.Random(10)
for _ in range(3000):
    a = "".join(rng.choice("abc") for _ in range(rng.randint(1, 7)))
    b = "".join(rng.choice("abc") for _ in range(rng.randint(1, 7)))
    assert smallest_merge(a, b) == smallest_brute(a, b), (a, b)
print("Bài 5: OK | ví dụ:", smallest_merge("cabaa", "bcaaa"), "| ('b','ba'):", smallest_merge("b", "ba"))

Bài 5: OK | ví dụ: bcaaacabaa | ('b','ba'): bab


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "merge lớn nhất theo từ điển":**

```python
i = j = 0; ans = []
while i < len(w1) and j < len(w2):
    if w1[i:] > w2[j:]: ans.append(w1[i]); i += 1
    else:               ans.append(w2[j]); j += 1
return "".join(ans) + w1[i:] + w2[j:]
```

**Checklist:**
- [ ] So sánh **cả hậu tố**, không chỉ ký tự đầu?
- [ ] Hàm so sánh tự viết xử lý đúng trường hợp **tiền tố** (`"ba" > "b"`)?
- [ ] Nối phần còn lại của **cả hai** chuỗi sau vòng lặp?
- [ ] Dùng list + `join` thay vì nối chuỗi lặp?
- [ ] Đã test: ký tự đầu bằng nhau, một chuỗi là tiền tố chuỗi kia, toàn một ký tự, chuỗi dài 3000?